# Pipeline Unifié de Détection de Fausses Nouvelles

## Vue d'Ensemble Complet

**Objectif:** Orchestrer un pipeline machine learning multi-branches pour la détection des fausses nouvelles:

1. **Style-Based Branch** (Analyse stylométrique): RoBERTa + RandomForest/XGBoost → **92% accuracy**
2. **Knowledge-Based Branch** (Vérification factuelle): Claim Detection + NLI Verification → **32% accuracy**
3. **Fusion Strategy** (Ensemble Learning): Combiner les 2 branches sur Part B → **?% accuracy (à déterminer)**

**Architecture Data:**
```
Dataset Complet (100%)
       ↓
   ┌───┴───┐
   ↓       ↓
 Part A   Part B
(80%)    (20%)
Training Fusion Opt+Test
   ↓
   ├─→ Style: Train + Test on Part A (92%)
   ├─→ Knowledge: Train + Test on Part A (32%)
   └─→ Fusion: Optimize weights on Part B validation, test on Part B test
```

**Timeline:**
- Style Branch (Phases 1-5): ~1-2 heures
- Knowledge Branch (Phases 1-5): ~1-2 heures
- Fusion Strategy (Phase 6): ~1 heure
- **Total: 3-5 heures** (GPU accelerated)

**Environnement:** Linux/Mac/Windows, GPU optionnelle (CPU très lent)

---

## Notes Importantes

✅ **Style & Knowledge déjà entraînés sur Part A** - Modèles gelés pour Phase 6  
⚠️ **Phase 6 utilise Part B uniquement** - Pas d'entraînement, seulement inférence + fusion optimization  
🔄 **Partition Part A/B unique** - Évite data leakage entre branches

Exécutez chaque section **dans l'ordre séquentiel**.

---

## Phase 0: Préparation Part A/B (Partition Unique)

Configuration initiale: création de la partition Part A/B unique pour toutes les branches.

**Important:** Une seule partition globale, pas de partitions par branche → évite data leakage.

- **Part A (80%):** Entraînement (Style + Knowledge)
- **Part B (20%):** Fusion optimization + testing (inférence only)

Exécutez cette cellule une seule fois au démarrage.

In [1]:
#!python unzip.py - optionnel si fichiers déjà décompressés
%cd data/
!python prepare_part_B_heterogeneous.py
%cd ..

print("\n✅ Phase 0 complétée: Part A/B partition créée")
print("   - Part A (80%): dataset_partA.csv, groundtruth_partA.csv, train_partA.jsonl")
print("   - Part B (20%): part_B_validation.csv (31.8k samples)")


/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/data
[INFO] 


[INFO] ██████████████████████████████████████████████████████████████████████
[INFO] PART A & B DATASET SPLIT - HETEROGENEOUS SOURCES
[INFO] ██████████████████████████████████████████████████████████████████████
[INFO] 
[INFO] PHASE 1: LOAD INPUTS & DONNÉES BRUTES
[INFO] ======================================================================
[INFO] 
1.1 Loading dataset.csv (LIAR+Twitter+UoVictoria)...
[INFO]      ✓ Loaded 61673 rows
[INFO]      Columns: ['text', 'label']
[INFO]      Label distribution: {0: 30863, 1: 30810}
[INFO] 
1.2 Loading groundtruth.csv (ClaimBuster)...
[INFO]      ✓ Loaded 1032 rows
[INFO]      Columns: ['Sentence_id', 'Text', 'Speaker', 'Speaker_title', 'Speaker_party', 'File_id', 'Length', 'Line_number', 'Sentiment', 'Verdict']
[INFO]      Verdict distribution: {-1: 731, 1: 238, 0: 63}
[INFO] 
1.3 Loading train.jsonl (FEVER)...
[INFO]      ✓ Loaded 145449 rows
[INFO]      Col

## Style based detection

### 1. Data preparation

If it has already been run, there is no need to run it again (but you can if you have 40 spare minutes)

## Phase 0: Part A & B Dataset Split from Heterogeneous Sources

**Purpose:** Create disjoint Part A (80%) and Part B (20%) datasets from 5 heterogeneous sources with unified label normalization.

**Strategy:**
- Load and normalize 3 main sources: dataset.csv (LIAR+Twitter+UoVictoria), groundtruth.csv (ClaimBuster), train.jsonl (FEVER)
- Filter invalid entries before sampling (Verdict==0, NOT_ENOUGH_INFO)
- Single stratified 80/20 split on unified data to ensure no overlap
- Extract Part A in 3 original formats for training separate branches
- Extract Part B as unified normalized CSV for fusion validation

**Output Files:**
- **Part A (80% training):**
  - `dataset_partA.csv`: LIAR+Twitter+UoVictoria for style-based training
  - `groundtruth_partA.csv`: ClaimBuster for claim detection training
  - `train_partA.jsonl`: FEVER for verification training
- **Part B (20% validation):**
  - `part_B_validation.csv`: Unified normalized dataset for fusion testing
  - `part_B_metadata.json`: Complete documentation and statistics

**Label Convention:**
- 0 = FAKE / FALSE / REFUTED
- 1 = TRUE / REAL / SUPPORTED

**Why Part A/B matters:**
Disjoint 80/20 split on unified normalized data ensures fair evaluation: Part A trains all three branches (Style, Claim Detection, Verification) while Part B validates fusion logic without data leakage.

In [2]:
%cd data/
!python data_extraction.py

!python prepare_part_B_heterogeneous.py
%cd ..


/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/data
Loading datasets...

DONE! Fusion of dataset have been saved to: dataset.csv

[INFO] 


[INFO] ██████████████████████████████████████████████████████████████████████
[INFO] PART A & B DATASET SPLIT - HETEROGENEOUS SOURCES
[INFO] ██████████████████████████████████████████████████████████████████████
[INFO] 
[INFO] PHASE 1: LOAD INPUTS & DONNÉES BRUTES
[INFO] ======================================================================
[INFO] 
1.1 Loading dataset.csv (LIAR+Twitter+UoVictoria)...
[INFO]      ✓ Loaded 61673 rows
[INFO]      Columns: ['text', 'label']
[INFO]      Label distribution: {0: 30863, 1: 30810}
[INFO] 
1.2 Loading groundtruth.csv (ClaimBuster)...
[INFO]      ✓ Loaded 1032 rows
[INFO]      Columns: ['Sentence_id', 'Text', 'Speaker', 'Speaker_title', 'Speaker_party', 'File_id', 'Length', 'Line_number', 'Sentiment', 'Verdict']
[INFO]      Verdict distribution: {-1: 731, 1: 238, 0: 63}
[INFO] 
1.3 L

In [3]:
%cd style_branch
!python feature_extraction_partA.py

!python print_features.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Loading data from ../data/splits/dataset_partA.csv...

Initializing NLP engine (spaCy + TextBlob/VADER)...
Loading linguistic models...
Extracting Style Metrics: 100%|███████████| 44470/44470 [18:26<00:00, 40.19it/s]

DONE! Features have been saved to: ../data/complete_train.csv
NEWS ANALYSIS:

Raw text:
ALERT! You must absolutely read this: https://www.google.com. The government is hiding 10,000 terrible and monstrous things! Wake up immediately @user! #news 😡

Loading linguistic models...
Normalized text:
ALERT! You must absolutely read this: [URL] The government is hiding [NB] terrible and monstrous things! Wake up immediately [MENTION]! #news 😡

Style vector (FEATURES):

has_hashtags                   : True
has_mentions                   : True
has_urls                       : True
has_numbers                    : True
total_characters               : 122
uppercase_ratio                : 0.082
fu

### 2. Fine tuning RoBERTa
Same as the previous cell, you don't need to run it, it can be very very very long because of your GPU (if you have it)
(MacBook Pro M4 with 20 GPU core (48 Go unified memory) -> 41 minutes)

In [4]:
%cd style_branch
!python split_data.py

!python fine_tunning.py

%run test_fine_tuned.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Loading the complete dataset (Style + Text)...

Data distribution:
Block A (RoBERTa Train) : 26682 rows
Block B (Training) : 8894 rows
Block C (Final Test)    : 8894 rows

Sanity Check - Class Distribution (Fake=1, True=0):
Block A (RoBERTa): 
label
0    0.548
1    0.452
Name: proportion, dtype: float64
Block B (XGBoost): 
label
0    0.548
1    0.452
Name: proportion, dtype: float64
Block C (Test):    
label
0    0.548
1    0.452
Name: proportion, dtype: float64

Splitting complete and files saved!
Nvidia GPU detected! Training will be hardware-accelerated.
Loading CSV files...
Word tokenization in progress... This may take a minute.
Loading weights: 100%|██████████████████████| 101/101 [00:00<00:00, 7622.85it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNE

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Analyzed sentence: 'The Federal Reserve announced a 0.5% increase in interest rates today.'
TRUE NEWS probability: 88.46%
FAKE NEWS probability: 11.54%

Analyzed sentence: 'BREAKING: Alien spaceship found hidden under the White House! The government is lying to us!!!'
TRUE NEWS probability: 5.68%
FAKE NEWS probability: 94.32%
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


### 3. Random Forest x XGBoost competition
If warnings appears about killing child process it is because you are on MacOs and it auto kill child process and the Python processus find any child process already kild by native MacOs processsus so it's just a warning and no action required. Others os aren't concerned.

In [5]:
%cd style_branch
!python model_comp.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Loading data (Style + RoBERTa Semantics)...

Configuration: 15 combinations tested per model (Cross-Validation x5).
Optimizing Random Forest...

Training Random Forest: 100%|███████████████████| 75/75 [00:08<00:00,  8.40it/s]
Finished in 0.15 min. Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 10, 'class_weight': 'balanced'}
Optimizing XGBoost...

Training XGBoost: 100%|█████████████████████████| 75/75 [00:06<00:00, 12.14it/s]
Finished in 0.1 min. Best parameters: {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.9}

Model                | Accuracy             | ROC-AUC (Quality)    | F1 Score             | Log Loss             |
------------------------------------------------------------------------------------------------------------------
Random Forest        |               92.10% |            

In [6]:
%cd style_branch
!python result_roberta.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Évaluation RoBERTa: 100%|█████████████████████| 556/556 [01:10<00:00,  7.86it/s]

RÉSULTATS DE ROBERTA SEUL (BASELINE)
Accuracy  : 92.11%
F1 Score  : 91.28%
ROC-AUC   : 98.25%
Log Loss  : 20.52%
--------------------------------------------------

Rapport détaillé :

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      4875
           1       0.91      0.91      0.91      4019

    accuracy                           0.92      8894
   macro avg       0.92      0.92      0.92      8894
weighted avg       0.92      0.92      0.92      8894

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


---

## KNOWLEDGE-BASED BRANCH (Phases 1-5)

**Objective:** Train DistilBERT claim detector + RoBERTa NLI verifier for knowledge-based fact-checking on Part A
- Architecture: Evidence retrieval (Google/Wolfram) → Entity extraction (spaCy) → NLI entailment scoring
- Core Models: DistilBERT claim detector + RoBERTa NLI verifier
- Baseline Accuracy: ~32% on Part A (identified issues: NEUTRAL class = 0%, needs ml_score activation)
- Frozen after Phase 5 for use in Phase 6 Fusion

**Note:** Cells below are extracted from knowledge_main.ipynb and adapted for unified execution.
Skip if models already trained (check: `knowledge_branch/claim_detector_model/`, `knowledge_branch/my_claim_model/`)

### Phase 1: Setup & Environment (Knowledge)

In [24]:
%cd knowledge_branch
!python setup_environment.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch
🚀 SETUP ENVIRONMENT - KNOWLEDGE BRANCH

🎮 Device : CUDA - NVIDIA GeForce RTX 4060
💾 GPU VRAM disponible : 8.2 GB

📂 Configuration des chemins...
   Répertoire de travail : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch
   ✅ Projet trouvé : True
   ✅ Data trouvée : True

📦 Vérification des dépendances...
   ✅ transformers
   ✅ torch
   ✅ spacy
   ✅ pandas
   ✅ datasets
   ✅ scikit-learn
   ❌ wikipedia-api - À installer

📥 Installation de 1 package(s)...
   ✅ Installation terminée

🔤 Téléchargement des modèles spaCy...
   ✅ en_core_web_sm
   ✅ fr_core_news_sm
   ✅ es_core_news_sm

📥 Téléchargement du dataset...
   ✅ Fichier déjà présent : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch/groundtruth.csv
🔍 Vérification des modules...
   ✅ evidence_retrieval
   ✅ claim_verification
   ✅ claim_detection

✅ SETUP TERMINÉ


### Phase 2: Claim Detector Training (Knowledge)

In [25]:
%cd knowledge_branch
!python train_claim_detector_partA.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch
🚀 TRAINING CLAIM DETECTOR (80% Part A)

📂 Chargement du dataset groundtruth_partA.csv (80% Part A)...
   📊 Dimensions : (780, 10)
   📋 Colonnes : ['Sentence_id', 'Text', 'Speaker', 'Speaker_title', 'Speaker_party', 'File_id', 'Length', 'Line_number', 'Sentiment', 'Verdict']
   📈 Distribution des labels :
Verdict
-1    590
 1    190
Name: count, dtype: int64
   ✅ Dataset préparé

✂️  Division du dataset...
   Train : 624 samples
   Test  : 156 samples

🤖 Fine-tuning du Claim Detector...
   Device : CPU (pour éviter les problèmes de mémoire GPU)

   tokenization en cours...
Map: 100%|██████████████████████████| 156/156 [00:00<00:00, 23000.26 examples/s]
   Chargement du modèle...
Loading weights: 100%|█████████████████████| 100/100 [00:00<00:00, 19192.39it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------

### Phase 3-4: Pipeline Initialization & NLI Verification (Knowledge)

In [26]:
%cd knowledge_branch
!python initialize_pipeline.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch
🚀 INITIALIZE KNOWLEDGE BRANCH

🔍 Initialisation du Evidence Retriever...
   ✅ Module evidence_retrieval importé
   Configuration :
      - Wolfram Alpha : ✅
      - Google API : ❌ (optionnel)
      - Google CSE : ✅
   ✅ Evidence Retriever initialisé

🔐 Initialisation du Claim Verifier...
   ✅ Module claim_verification importé
Loading weights: 100%|██████████████████████| 202/202 [00:00<00:00, 8348.27it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
   ✅ Claim Verifier initialisé

💾 Sauvegarde de la configuration...
   ✅ Configuration définie

🧪 Test du Evidence Retriever...
 

### Phase 5: Evaluation & Demo (Knowledge)

In [27]:
%cd knowledge_branch
!python evaluate_pipeline_partA.py
!python full_pipeline.py

from full_pipeline import test_multiple_languages

test_multiple_languages()

%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch
🚀 EVALUATION - KNOWLEDGE BRANCH PIPELINE (80% Part A)

✅ Modules importés

📂 Chargement du dataset FEVER train_partA.jsonl (80% Part A)...
   📊 Total : 81962 instances
   📈 Distribution des labels :
label
SUPPORTED                    59081
REFUTED                      22631
NEUTRAL / NOT ENOUGH INFO      250
Name: count, dtype: int64

⚖️  Équilibrage du dataset (30 par classe)...
   ✅ Dataset équilibré : 90 instances
   📈 Distribution :
label
REFUTED                      30
SUPPORTED                    30
NEUTRAL / NOT ENOUGH INFO    30
Name: count, dtype: int64

🔧 Initialisation des composants...
Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 18149.77it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.posit

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   ✅ Pipeline initialisé


--- EN ---

🚀 --- TRAITEMENT DU TEXTE (EN) ---

[1] The Great Wall of China is one of the largest structures.
    ℹ️  ML Score: 0.994
    🔍 Vérification en cours...
    ✅ Source: Great Wall of China
    📄 Preuve: The Great Wall of China (traditional Chinese: 萬里長城; simplified Chinese: 万里长城; pi...
    ✅ Verdict: NEUTRAL / NOT ENOUGH INFO (confiance: 0.998)

[2] Paris is in France.

    🏷️  Entités: GPE: Paris, France
    🔍 Vérification en cours...
    ✅ Source: WolframAlpha
    📄 Preuve: Paris is the capital and largest city of France, with an estimated city populati...
    ✅ Verdict: SUPPORTED (confiance: 0.999)


📊 RÉSUMÉ DU RAPPORT

Total d'items traités : 2

Verdicts :
  NEUTRAL / NOT ENOUGH INFO :   1 ( 50.0%)
  SUPPORTED            :   1 ( 50.0%)

Détails :
  1. [NEUTRAL / NOT ENOUGH INFO] The Great Wall of China is one of the largest structures.... (conf: 0.998)
  2. [SUPPORTED      ] Paris is in France.... (conf: 0.999)


--- FR ---

🚀 --- TRAITEMENT DU

---

# PHASE 6: FUSION BRANCH - SEQUENTIAL STRATEGY COMPARISON

**Objectif:** Comparer 5 stratégies de fusion sur Part B (données non vues) en exécution séquentielle.

**Architecture:**
```
Part B (31.8k samples) - UNSEEN DATA
      ↓
  Split 50/50
 /            \
Fusion Train   Fusion Test
(15.9k)        (15.9k)
   ↓              ↓
Optimize (1s   Evaluate (1s
à la fois)     à la fois)
```

**Stratégies testées (séquentiellement):**
1. **Cascading**: If Style confiant → Style, else Knowledge
2. **Confidence-Weighted**: Poids fixes (0.92 vs 0.32)
3. **Disagreement-Adaptive**: Adaptation selon accord/désaccord
4. **Weighted + Threshold** ⭐: Gridsearch 125 configs
5. **Stacked RF** ⭐: Meta-learner RandomForest

**Résultats attendus:**
- Style baseline: 92% F1
- Knowledge baseline: 32% F1
- Fusion: 88-91% F1


# Fusion Branch - Simplified Pipeline

This notebook runs the complete fusion pipeline by calling individual Python scripts.
Each cell is self-contained and executes a specific phase of the fusion analysis.

**Pipeline Overview:**
1. Verify models exist
2. Load predictions from frozen models
3. Split Part B into train/test
4-8. Run 5 fusion strategies
9. Compare and visualize results

date: 2026-04-12

## Section 0: Verify Frozen Models

In [ ]:
import subprocess
from pathlib import Path
import sys

# Change to fusion_branch and run script
%cd fusion_branch
!python 00_verify_models.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch
Section 1: VÉRIFICATION MODÈLES GELÉS (Part A Training)
✅ Style best_model              : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch/results/best_model.pkl
✅ Style RoBERTa                 : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch/roberta_fine_tunned
✅ Knowledge Claim Detector      : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch/claim_detector_model
✅ Knowledge NLI                 : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch/my_claim_model
✅ TOUS LES MODÈLES PRÉSENTS - Prêt pour Phase 6B/6C

✅ Dossier results créé pour stockage résultats
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 1: Load Predictions from Frozen Models

In [ ]:
%cd fusion_branch
!python 01_load_predictions.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 2: CHARGER PART B ET GÉNÉRER PRÉDICTIONS BRUTES
✅ Configuration fusion RÉVISÉE chargée
   - 5 Stratégies à tester
   - Stratégie 4: 150 configurations
   - Device: cpu

FUSION PHASE 6B: Charger Modèles Gelés + Générer Prédictions Part B
✅ evidence_loader exécuté avec succès

✅ Prédictions chargées:
   - Style predictions: 31804 samples
   - Knowledge predictions: 31804 samples
   - Ground truth labels: 31804 samples
   - Classe 0 (FAKE): 11937 (37.5%)
   - Classe 1 (TRUE): 19867 (62.5%)
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 2: Split Part B Data

In [ ]:
%cd fusion_branch
!python 02_split_data.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 3: SPLIT PART B EN FUSION_TRAIN/TEST (50/50)

📊 Split Results:
   Fusion Train: 15902 samples (Classe 0: 5968, Classe 1: 9934)
   Fusion Test:  15902 samples (Classe 0: 5969, Classe 1: 9933)

✅ Données de fusion sauvegardées:
   - fusion_train.csv: 15902 rows
   - fusion_test.csv: 15902 rows
   - part_b_split.pkl: données splits pour stratégies
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 3: Strategy 1 - Cascading (Style First)

In [ ]:
%cd fusion_branch
!python 03_strategy_1.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 4: STRATÉGIE 1 - CASCADING (Style First)

🔍 Gridsearch sur fusion_train...
✅ Meilleur seuil trouvé: 0.50 (F1 train: 0.6013)

📊 Résultats Stratégie 1 (Test unseen):
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035
✅ Résultats sauvegardés: strategy_1_cascading_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 4: Strategy 2 - Confidence-Weighted Voting

In [ ]:
%cd fusion_branch
!python 04_strategy_2.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch
✅ Configuration fusion RÉVISÉE chargée
   - 5 Stratégies à tester
   - Stratégie 4: 150 configurations
   - Device: cpu

Section 5: STRATÉGIE 2 - CONFIDENCE-WEIGHTED VOTING

📊 Résultats Stratégie 2 (Test unseen):
   Accuracy:  0.4357
   Precision: 0.5500
   Recall:    0.5309
   F1-Score:  0.5403
   (Note: pas de gridsearch - poids fixes 0.92/0.32)
✅ Résultats sauvegardés: strategy_2_cf_voting_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 5: Strategy 3 - Disagreement-Adaptive Weighting

In [ ]:
%cd fusion_branch
!python 05_strategy_3.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 6: STRATÉGIE 3 - DISAGREEMENT-ADAPTIVE WEIGHTING

🔍 Gridsearch sur fusion_train...
✅ Meilleur poids trouvé: 0.30 (F1 train: 0.6013)

📊 Résultats Stratégie 3 (Test unseen):
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035
✅ Résultats sauvegardés: strategy_3_disagreement_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 6: Strategy 4 - Weighted Voting + Threshold ⭐

In [ ]:
%cd fusion_branch
!python 06_strategy_4.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 7: STRATÉGIE 4 - WEIGHTED + THRESHOLD ⭐

🔍 Gridsearch sur fusion_train (125 configurations)...
✅ Configurations testées: 125
✅ Meilleurs paramètres trouvés:
   w_style=0.50, w_knowledge=0.45, threshold=0.40
   F1 train: 0.6013

📊 Résultats Stratégie 4 (Test unseen):
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035
✅ Résultats sauvegardés: strategy_4_weighted_threshold_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 7: Strategy 5 - Stacked RandomForest ⭐

In [ ]:
%cd fusion_branch
!python 07_strategy_5.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch
✅ Configuration fusion RÉVISÉE chargée
   - 5 Stratégies à tester
   - Stratégie 4: 150 configurations
   - Device: cpu

Section 8: STRATÉGIE 5 - STACKED RANDOMFOREST ⭐

🔧 Entraînement du meta-learner RandomForest...
✅ Stacked RF entraîné sur 15902 samples
✅ Entraînement complété

📊 Résultats Stratégie 5 (Test unseen):
   Accuracy:  0.7706
   Precision: 0.7350
   Recall:    0.9894
   F1-Score:  0.8435

🔍 Feature Importance (Meta-learner):
   style_pred:     0.1485
   style_conf:     0.7909
   knowledge_pred: 0.0009
   knowledge_conf: 0.0597
✅ Résultats sauvegardés: strategy_5_stacked_rf_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 8: Final Comparison & Visualization

In [ ]:
%cd fusion_branch
!python 08_comparison_visualize.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 9-10: COMPARAISON FINALE & VISUALISATION

📊 TABLEAU COMPARATIF (classé par F1-Score)
             Strategy  Accuracy  Precision   Recall  F1-Score
           Stacked RF  0.770595   0.735024 0.989429  0.843460
Cascading Style First  0.465350   0.562158 0.651465  0.603525
Disagreement-Adaptive  0.465350   0.562158 0.651465  0.603525
   Weighted+Threshold  0.465350   0.562158 0.651465  0.603525
  Confidence-Weighted  0.435668   0.550016 0.530857  0.540266

✅ Tableau comparatif sauvegardé: comparison_table.csv

🏆 MEILLEURE STRATÉGIE: Stacked RF
   F1-Score (Part B unseen test): 0.8435

PHASE 6: FUSION BRANCH - RAPPORT D'ANALYSE FINAL

DATE: 2026-04-12 (Automated execution)
DATASET: Part B (31.8k samples) - unseen validation & test data

RÉSULTATS PAR STRATÉGIE (Part B unseen test)

1. Stacked RF
   Accuracy:  0.7706
   Precision: 0.7350
   Recall:    0.9894
   F1-Score:  0.8435

2. Cascading Sty

---

## 📊 SUMMARY & COMPARATIVE ANALYSIS

**Complete Pipeline Execution Map:**

```
Phase 0 (Unified) →  Part A/B Partition Creation (80/20 split)
                ↓
            ┌────────────────────────────────────────┐
            ↓                                        ↓
    Style Branch (Phase 1-5)         Knowledge Branch (Phase 1-5)
    Part A Training                   Part A Training
    • Data ingestion + Features       • Claim detection + NLI verification
    • RoBERTa fine-tuning            • Evidence retrieval + Entity extraction
    • RF vs XGB ensemble             • Pipeline initialization
    RESULT: 92% accuracy ✅          RESULT: 32% accuracy ⚠️
                ↓                                        ↓
            └────────────────────────────────────────┘
                              ↓
        Phase 6: Fusion (Part B) - 5 Stratégies
        ────────────────────────────────────────
        1. Cascading (Style First) → F1: 0.6035
        2. Confidence-Weighted → F1: 0.5403
        3. Disagreement-Adaptive → F1: 0.6035
        4. Weighted + Threshold → F1: 0.6035
        5. Stacked RandomForest ⭐ → F1: 0.8435
        ────────────────────────────────────────

        BEST: Stacked RF (F1: 0.8435, +39.8% vs Cascading)```

                              ↓              Tableau Comparatif & Recommandations

In [30]:
import json
import pandas as pd
from pathlib import Path

print("\n" + "="*70)
print("📊 TABLEAU COMPARATIF: TOUTES LES STRATÉGIES")
print("="*70)

# Charger le tableau de comparaison des 5 stratégies
comparison_file = Path('fusion_branch/results/comparison_table.csv')
if comparison_file.exists():
    df_comparison = pd.read_csv(comparison_file)
    print("\n" + df_comparison.to_string(index=False))
else:
    print("⚠️  Fichier comparison_table.csv non trouvé")

# Résumé des branches principales
print("\n" + "="*70)
print("RÉSUMÉ FINAL: COMPARAISON COMPLÈTE")
print("="*70)

results_data = {
    'Branch': ['Style-Based (Part A)', 'Knowledge-Based (Part A)', 'Best Fusion (Part B)'],
    'F1-Score': [0.92, 0.32, 0.8435],
    'Architecture': ['RoBERTa + RF/XGB', 'DistilBERT + RoBERTa NLI', 'Stacked RandomForest'],
    'Status': ['✅ Baseline', '⚠️  Weak', '⭐ RECOMMENDED'],
}

df_main = pd.DataFrame(results_data)
print("\n" + df_main.to_string(index=False))

print("\n" + "="*70)
print("✅ PIPELINE UNIFIÉ COMPLÈTEMENT EXÉCUTÉ")
print("="*70)
print("\nToutes les phases ont été orchestrées avec succès:")
print("  ✅ Phase 0: Part A/B partition (80/20 split)")
print("  ✅ Phases 1-5 Style: Entraînement RoBERTa + Ensemble")
print("  ✅ Phases 1-5 Knowledge: Entraînement Claim Detector + NLI")
print("  ✅ Phase 6: Fusion (5 stratégies testées sur Part B)")
print("\nRésultats disponibles dans fusion_branch/results/")
print("  📊 comparison_table.csv")
print("  📄 best_strategy_analysis.txt")
print("  📈 fusion_strategy_comparison.png")


📊 TABLEAU COMPARATIF: TOUTES LES STRATÉGIES

             Strategy  Accuracy  Precision   Recall  F1-Score
           Stacked RF  0.770595   0.735024 0.989429  0.843460
Cascading Style First  0.465350   0.562158 0.651465  0.603525
Disagreement-Adaptive  0.465350   0.562158 0.651465  0.603525
   Weighted+Threshold  0.465350   0.562158 0.651465  0.603525
  Confidence-Weighted  0.435668   0.550016 0.530857  0.540266

RÉSUMÉ FINAL: COMPARAISON COMPLÈTE

                  Branch  F1-Score             Architecture        Status
    Style-Based (Part A)    0.9200         RoBERTa + RF/XGB    ✅ Baseline
Knowledge-Based (Part A)    0.3200 DistilBERT + RoBERTa NLI      ⚠️  Weak
    Best Fusion (Part B)    0.8435     Stacked RandomForest ⭐ RECOMMENDED

✅ PIPELINE UNIFIÉ COMPLÈTEMENT EXÉCUTÉ

Toutes les phases ont été orchestrées avec succès:
  ✅ Phase 0: Part A/B partition (80/20 split)
  ✅ Phases 1-5 Style: Entraînement RoBERTa + Ensemble
  ✅ Phases 1-5 Knowledge: Entraînement Claim Detector + N

In [33]:
# ===== ANALYSE DÉTAILLÉE: AFFICHER LES RÉSULTATS COMPLETS =====

from pathlib import Path
import json

print("\n" + "="*70)
print("📄 ANALYSE DÉTAILLÉE: PERFORMANCE DES 5 STRATÉGIES")
print("="*70)

# Charger et afficher le rapport d'analyse
analysis_file = Path("fusion_branch/results/best_strategy_analysis.txt")
if analysis_file.exists():
    with open(analysis_file, 'r', encoding='utf-8') as f:
        analysis_content = f.read()
        # Afficher seulement les parties clés
        lines = analysis_content.split('\n')
        for i, line in enumerate(lines):
            if 'RÉSULTATS' in line or 'RECOMMANDATION' in line or 'MEILLEURE' in line:
                print('\n'.join(lines[max(0,i-2):min(len(lines),i+15)]))
                break
else:
    print("⚠️  Fichier analysis non trouvé")

print("\n" + "="*70)
print("🎯 RECOMMANDATION FINALE")
print("="*70)
print("""
✅ MEILLEURE STRATÉGIE: Stacked RandomForest

   • F1-Score: 0.8435 (Part B unseen test)
   • Accuracy: 0.7706 (77.1%)
   • Recall: 0.9894 (99% de détection des fakes)
   • Amélioration: +39.8% vs Cascading

🔑 KEY INSIGHT:
   → Style confidence est le facteur DOMINANT (79% feature importance)
   → Le meta-learner apprend à combiner optimalement les deux modèles
   → Excellent équilibre Precision/Recall pour production

💡 UTILISATION:
   → Production: Utiliser Stacked RandomForest
   → Baseline: Cascading ou Confidence-Weighted
   → Explainabilité: Disagreement-Adaptive
""")

print("\n" + "="*70)
print("📁 FICHIERS DE SORTIE GÉNÉRÉS")
print("="*70)
print("""
fusion_branch/results/
├── comparison_table.csv              # Tableau récapitulatif 5 stratégies
├── best_strategy_analysis.txt        # Rapport détaillé + recommendations
├── fusion_strategy_comparison.png    # Graphiques comparatifs
├── strategy_1_cascading_report.json  # Détails Stratégie 1
├── strategy_2_cf_voting_report.json  # Détails Stratégie 2
├── strategy_3_disagreement_report.json # Détails Stratégie 3
├── strategy_4_weighted_threshold_report.json # Détails Stratégie 4
├── strategy_5_stacked_rf_report.json # Détails Stratégie 5 ⭐
├── fusion_train.csv                  # Données fusion train (15.9k)
└── fusion_test.csv                   # Données fusion test (15.9k)
""")


📄 ANALYSE DÉTAILLÉE: PERFORMANCE DES 5 STRATÉGIES

RÉSULTATS PAR STRATÉGIE (Part B unseen test)

1. Stacked RF
   Accuracy:  0.7706
   Precision: 0.7350
   Recall:    0.9894
   F1-Score:  0.8435

2. Cascading Style First
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035


🎯 RECOMMANDATION FINALE

✅ MEILLEURE STRATÉGIE: Stacked RandomForest

   • F1-Score: 0.8435 (Part B unseen test)
   • Accuracy: 0.7706 (77.1%)
   • Recall: 0.9894 (99% de détection des fakes)
   • Amélioration: +39.8% vs Cascading

🔑 KEY INSIGHT:
   → Style confidence est le facteur DOMINANT (79% feature importance)
   → Le meta-learner apprend à combiner optimalement les deux modèles
   → Excellent équilibre Precision/Recall pour production

💡 UTILISATION:
   → Production: Utiliser Stacked RandomForest
   → Baseline: Cascading ou Confidence-Weighted
   → Explainabilité: Disagreement-Adaptive


📁 FICHIERS DE SORTIE GÉNÉRÉS

fusion_branch/results/
├── comparison_table.csv             